In [5]:
import os

from dataclasses_json import config
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver
from rich import print

from langchain.chat_models import init_chat_model
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware, HumanInTheLoopMiddleware
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver

# 環境変数を読み込む
load_dotenv(override=True)

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_API_BASE = os.getenv("OPENROUTER_API_BASE")

# 要約生成用のモデルを初期化
model = init_chat_model(
    model="openai/gpt-5.4-mini",
    model_provider="openai",
    profile={"max_input_tokens": 128_000},
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_API_BASE,
)

tools

In [6]:
from langchain.tools import tool


@tool
def get_weather(city: str, is_forcast: bool = False) -> str:
    """
    指定された都市の天気を照会する
    
    Args:
        city: 都市名
        is_forcast: 明日の天気予報を含めるかどうか？
    """
    res = f"{city}は今日良い天気です"
    if is_forcast:
        res += "\n明日は雨です"
    return res


@tool
def get_news() -> str:
    """
    当日のニュースを照会する
    """
    return "日本のタンカー3隻がホルムズ海峡を通過"


@tool
def read_email_tool(email_id: str) -> str:
    """メールIDからメール内容を読み取る疑似関数"""
    return f"メールID：{email_id}\n内容は空です"


@tool
def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """メール送信の疑似関数"""
    print(">>> 実際にメール送信ツールが実行されました")
    return f"{recipient} に送信したメールの件名は：{subject}、本文：{body}"


In [7]:
agent = create_agent(
    model=model,
    tools=[get_weather, get_news, read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "get_weather": True,
                "get_news": True,
                "read_email_tool": False,
                "send_email_tool": {
                    "allowed_decisions": ["approve", "reject"],
                    "description": "メール送信が中断されました"
                },
            },
            description_prefix="中断されました"
        ),
    ],
)
# 会話の一意性を保つため
config = {"configurable": {"thread_id": "1"}}
response = agent.invoke({
    "messages": [
        HumanMessage(content="今日の北京の天気を調べてください"
                             "今日のニュースを調べてください"
                             "ID 'sk2131421' のメール内容を確認してください、"
                             "15641685664@qq.com にメールを送信してください、件名は'ははは'、内容は：'こんにちは'"
                             "この4つを同時に行ってください")
    ],
},
    config=config
)
print("==== 1回目の invoke の戻り値 ====")
print("========= 生のレスポンス =========")
print(response)
print("========= 整形出力 =========")
for msg in response["messages"]:
    msg.pretty_print()
# ポイント：中断情報を確認
interrupts = response.get("__interrupt__", [])
print("========== interrupts ==========")
print(interrupts)
# print("==== interrupt リクエストを一つずつ出力 ====")
action_requests = interrupts[0].value["action_requests"]
for action_request in action_requests:
    print(action_request)


==== 1回目の invoke の戻り値 ====

========= 生のレスポンス =========

{
    'messages': [
        HumanMessage(
            content="今日の北京の天気を調べてください今日のニュースを調べてくださいID 'sk2131421' 
のメール内容を確認してください、15641685664@qq.com 
にメールを送信してください、件名は'ははは'、内容は：'こんにちは'この4つを同時に行ってください",
            additional_kwargs={},
            response_metadata={},
            id='9b1fd86b-4f55-4c20-be68-b7cff75d17f8'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 100,
                    'prompt_tokens': 239,
                    'total_tokens': 339,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': None,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': None,
                        'image_tokens': 0
                    },
                    'prompt_tokens_details': {
                        'audio_tokens': 0,
                        'cached_tokens': 0,
                        'cache_write_tokens': 0,
                        'video_tokens': 0
                    },
                    'cost': 0.00062925,
                    'is_byok': False,
                    'cost_details': {
                        'upstream_inference_cost': 0.00062925,
                        'upstream_inference_prompt_cost': 0.00017925,
                        'upstream_inference_completions_cost': 0.00045
                    }
                },
                'model_provider': 'openai',
                'model_name': 'openai/gpt-5.4-mini',
                'system_fingerprint': None,
                'id': 'gen-1786070574-FogcsjBVWwm3BmvITkgK',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--019fda1a-2401-7e30-8476-febc36b27e78-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'city': '北京', 'is_forcast': False},
                    'id': 'call_8mZHlE1qJ8s3xgC59piD1m8Y',
                    'type': 'tool_call'
                },
                {'name': 'get_news', 'args': {}, 'id': 'call_kj4dXSJvLwUFPwFFW5RBZFMl', 'type': 'tool_call'},
                {
                    'name': 'read_email_tool',
                    'args': {'email_id': 'sk2131421'},
                    'id': 'call_pE4xsh6C4FK8SbEEPHEYY6px',
                    'type': 'tool_call'
                },
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': 'ははは', 'body': 'こんにちは'},
                    'id': 'call_xt7OA6FWeRLxlV3mpF5BHDbb',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 239,
                'output_tokens': 100,
                'total_tokens': 339,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        )
    ],
    '__interrupt__': [
        Interrupt(
            value={
                'action_requests': [
                    {
                        'name': 'get_weather',
                        'args': {'city': '北京', 'is_forcast': False},
                        'description': "中断されました\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': 
False}"
                    },
                    {'name': 'get_news', 'args': {}, 'description': '中断されました\n\nTool: get_news\nArgs: {}'},
                    {
                        'name': 'send_email_tool',
                        'args': {'recipient': '15641685664@qq.com', 'subject': 'ははは', 'body': 'こんにちは'},
                        'description': 'メール送信が中断されました'
                    }
                ],
                'review_conf

========= 整形出力 =========

================================ Human Message =================================

今日の北京の天気を調べてください今日のニュースを調べてくださいID 'sk2131421' のメール内容を確認してください、15641685664@qq.com にメールを送信してください、件名は'ははは'、内容は：'こんにちは'この4つを同時に行ってください
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_8mZHlE1qJ8s3xgC59piD1m8Y)
 Call ID: call_8mZHlE1qJ8s3xgC59piD1m8Y
  Args:
    city: 北京
    is_forcast: False
  get_news (call_kj4dXSJvLwUFPwFFW5RBZFMl)
 Call ID: call_kj4dXSJvLwUFPwFFW5RBZFMl
  Args:
  read_email_tool (call_pE4xsh6C4FK8SbEEPHEYY6px)
 Call ID: call_pE4xsh6C4FK8SbEEPHEYY6px
  Args:
    email_id: sk2131421
  send_email_tool (call_xt7OA6FWeRLxlV3mpF5BHDbb)
 Call ID: call_xt7OA6FWeRLxlV3mpF5BHDbb
  Args:
    recipient: 15641685664@qq.com
    subject: ははは
    body: こんにちは


========== interrupts ==========

[
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'get_weather',
                    'args': {'city': '北京', 'is_forcast': False},
                    'description': "中断されました\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': 
False}"
                },
                {'name': 'get_news', 'args': {}, 'description': '中断されました\n\nTool: get_news\nArgs: {}'},
                {
                    'name': 'send_email_tool',
                    'args': {'recipient': '15641685664@qq.com', 'subject': 'ははは', 'body': 'こんにちは'},
                    'description': 'メール送信が中断されました'
                }
            ],
            'review_configs': [
                {'action_name': 'get_weather', 'allowed_decisions': ['approve', 'edit', 'reject']},
                {'action_name': 'get_news', 'allowed_decisions': ['approve', 'edit', 'reject']},
                {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'reject']}
            ]
        },
        id='8c301c6ac0dc829c1311724b9d5d2e9f'
    )
]

{
    'name': 'get_weather',
    'args': {'city': '北京', 'is_forcast': False},
    'description': "中断されました\n\nTool: get_weather\nArgs: {'city': '北京', 'is_forcast': False}"
}

{'name': 'get_news', 'args': {}, 'description': '中断されました\n\nTool: get_news\nArgs: {}'}

{
    'name': 'send_email_tool',
    'args': {'recipient': '15641685664@qq.com', 'subject': 'ははは', 'body': 'こんにちは'},
    'description': 'メール送信が中断されました'
}

In [8]:
from langgraph.types import Command

# 中断があれば、Human-in-the-loop に入ったことを意味する
weather_decision = {
    "type": "edit",
    "edited_action": {
        "name": "get_weather",
        "args": {"city": "中国上海市", "is_forcast": True}
    }
}

news_decision = {
    "type": "approve",
}
send_email_decision = {
    "type": "approve"
}
decisions = {
    "decisions": []
}

# 決定の順序は、返された中断リクエストの順序と一致している必要がある
for action_request in action_requests:
    if action_request["name"] == "get_weather":
        decisions["decisions"].append(weather_decision)
    if action_request["name"] == "get_news":
        decisions["decisions"].append(news_decision)
    if action_request["name"] == "send_email_tool":
        decisions["decisions"].append(send_email_decision)

agent.invoke(
    Command(resume=decisions),
    config=config
)

>>> 実際にメール送信ツールが実行されました

{'messages': [HumanMessage(content="今日の北京の天気を調べてください今日のニュースを調べてくださいID 'sk2131421' のメール内容を確認してください、15641685664@qq.com にメールを送信してください、件名は'ははは'、内容は：'こんにちは'この4つを同時に行ってください", additional_kwargs={}, response_metadata={}, id='9b1fd86b-4f55-4c20-be68-b7cff75d17f8'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 239, 'total_tokens': 339, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 0.00062925, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 0.00062925, 'upstream_inference_prompt_cost': 0.00017925, 'upstream_inference_completions_cost': 0.00045}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-5.4-mini', 'system_fingerprint': None, 'id': 'gen-1786070574